# Controlling MODFLOW 6 with the API — C. Adjusting recharge with a callback

This is part C of the MODFLOW 6 API series — see [**A. Basic usage**](mf6-api-a-basic.ipynb) for the API lifecycle and the callback mechanism.

Modify model inputs while the simulation is running. A `modflowapi` callback intercepts the start of each stress period and multiplies the recharge by 1.1, increasing precipitation while the model runs without rewriting the input files. Then confirm the change directly from the model's own budget output.

Run the imports and library-location cells below first.

## What this notebook covers

Change a model input while the simulation is running, using a callback, and
confirm from the model's own output that the change took.

By the end of this notebook you will be able to:

- write a callback and hand it to `modflowapi.run_simulation()`,
- change recharge at the start of every stress period without touching the input
  files, and
- confirm the change from the budget the run wrote, with no second run.


## Imports and setup

Import FloPy, the `modflowapi` interface, and the supporting plotting and data libraries, then create a directory for the figures saved by this notebook.

In [ ]:
%matplotlib inline

import pathlib as pl

import flopy
import matplotlib.pyplot as plt
import numpy as np
from mf6_notebook_helpers import find_mf6_libraries, to_display, unit
from modflowapi import Callbacks, run_simulation

In [ ]:
fig_path = pl.Path(".figures")
fig_path.mkdir(exist_ok=True)

In [ ]:
sample_frequency = "annual"  # monthly or annual
name = "sv"

### Locate the MODFLOW 6 library

The API drives MODFLOW 6 through its compiled shared library (`libmf6`), not the `mf6` command-line executable. The `find_mf6_libraries` helper in `mf6_notebook_helpers.py` locates the library (`.so` on Linux, `.dylib` on macOS, `.dll` on Windows) and the `mf6` executable inside the active pixi environment; confirm that both exist.

In [ ]:
# libmf6 (the shared library the API drives) and the mf6 executable,
# located in the active pixi environment
lib_name, mf6_exe = find_mf6_libraries()

In [ ]:
lib_name.is_file(), mf6_exe.is_file()

## Increase precipitation with a callback

Modify model inputs while the simulation is running. The `callback_function` below is called by `modflowapi` at key points in the solution (for example `Callbacks.initialize`, `Callbacks.stress_period_start`, and `Callbacks.iteration_start`); here it intercepts the start of each stress period and multiplies the recharge by 1.1, increasing precipitation while the model runs without rewriting the input files.

The callback is the most convenient way to work with the standard stress packages. `modflowapi` exposes the running model as familiar Python objects - the simulation (`sim`), each model (`sim.sv`), and its packages (`ml.rch_0`) - so you can reach a package by name and read or change its `stress_period_data` directly with numpy-style syntax (`spd["recharge"] *= 1.1`). The edits take effect in the running model immediately, and you never have to know the internal MODFLOW variable names, memory addresses, or array layout.

For data the package interface does not expose - internal solver variables or advanced-package terms, for example - you use the lower-level `get_value` / `get_value_ptr` memory access instead. The other API notebooks rely on it: notebook **B** reads the solver's convergence variable that way, and notebook **E** reads the simulated stream (SFR) flow and sets a prediction-well rate from it.

In [ ]:
ws = pl.Path(f"../data/synthetic-valley/synthetic-valley-base-{sample_frequency}")
new_ws = pl.Path(f"models/synthetic-valley-base-{sample_frequency}_api_c")

sim = flopy.mf6.MFSimulation.load(
    sim_name=name,
    sim_ws=ws,
    write_headers=False,
)
sim.set_sim_path(new_ws)
sim.write_simulation()

In [ ]:
def callback_function(sim, callback_step):
    """
    A demonstration function that dynamically adjusts recharge in a
    MODFLOW 6 model through the MODFLOW-API

    Parameters
    ----------
    sim : modflowapi.ApiSimulation
        A simulation object for the solution group that is
        currently being solved
    step : enumeration
        modflowapi.Callbacks enumeration object that indicates
        the part of the solution modflow is currently in.
    """
    ml = sim.sv
    if callback_step == Callbacks.initialize:
        print(sim.models)

    if callback_step == Callbacks.stress_period_start:
        rcha = ml.rch_0
        spd = rcha.stress_period_data
        print(f"updating recharge: stress_period={ml.kper + 1}")
        print(spd)
        spd["recharge"] *= 1.1

    if callback_step == Callbacks.timestep_start:
        pass

    if callback_step == Callbacks.iteration_start:
        # we can implement complex solutions to boundary conditions here!
        pass

In [ ]:
run_simulation(lib_name, new_ws, callback_function, verbose=False)

### Confirm the callback changed the results

Running a callback is only convincing if you can see its effect. The cell below reads the **total recharge** over the whole model from the volumetric budget (`BUDGETCSV`) that the run above just wrote, and compares it with the recharge the RCH input specifies.

There is no need to run the model a second time to get the baseline. The callback multiplies recharge by 1.1 in every stress period, so the original recharge is the increased recharge divided by 1.1, and both curves come from this single run. (If a callback instead applied a non-uniform change, you would record the before/after values inside the callback itself, or compare with a separately saved baseline budget.) The two curves stay a constant 10 percent apart by construction. What confirms the callback is the ratio of the recharge the run reports to the recharge in the RCH input: 1.1 if the callback ran, 1.0 if it did not.

In [ ]:
# total recharge (m3/d) from the budget CSV the callback run just wrote; the
# callback scaled it by 1.1, so the original recharge is that divided by 1.1
gwf = sim.get_model(name)
budget = gwf.output.budgetcsv().data
rch_increased = budget["RCH(RCH_0)_IN"] - budget["RCH(RCH_0)_OUT"]
rch_baseline = rch_increased / 1.1

# total recharge the RCH input specifies (m3/d), rate times cell area
rch = gwf.get_package("rch_0")
area = np.outer(gwf.dis.delc.array, gwf.dis.delr.array)
rch_input = []
for kper in range(sim.tdis.nper.array):
    spd = rch.stress_period_data.get_data(kper)
    rows = [cellid[1] for cellid in spd["cellid"]]
    cols = [cellid[2] for cellid in spd["cellid"]]
    rch_input.append((spd["recharge"] * area[rows, cols]).sum())
ratio = rch_increased / np.array(rch_input)
print(f"simulated / input recharge: {ratio.min():.4f} to {ratio.max():.4f}")
print(
    f"stress period 1: {to_display(rch_baseline[0], 'flow'):,.0f} to"
    f" {to_display(rch_increased[0], 'flow'):,.0f} {unit('flow', plain=True)}"
)

period = np.arange(1, len(budget) + 1)
with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
    ax.plot(
        period,
        to_display(rch_baseline, "flow"),
        marker="o",
        ms=4,
        lw=1.5,
        color="0.4",
        drawstyle="steps-mid",
        label="Original recharge",
    )
    ax.plot(
        period,
        to_display(rch_increased, "flow"),
        marker="o",
        ms=4,
        lw=1.5,
        color="blue",
        drawstyle="steps-mid",
        label="Increased recharge (callback, +10%)",
    )
    flopy.plot.styles.heading(
        ax=ax,
        heading="Total recharge",
    )
    flopy.plot.styles.xlabel(
        ax=ax,
        label="Stress period",
    )
    flopy.plot.styles.ylabel(
        ax=ax,
        label=f"Recharge rate, {unit('flow')}",
    )
    flopy.plot.styles.remove_edge_ticks(
        ax=ax,
    )
    ax.set_xticks(period[::2])
    ax.yaxis.set_major_formatter(plt.matplotlib.ticker.StrMethodFormatter("{x:,.0f}"))
    ax.legend(loc="best", frameon=False)
    fig.savefig(fig_path / "sv-recharge-callback.png", dpi=300, transparent=False)

**What to look for.** The two curves track each other with a constant 10 percent
gap, which they must: the baseline is the increased recharge divided by 1.1. The
confirmation that the callback fired is the ratio the run itself reports.
Simulated recharge is 1.1000 times the recharge the RCH package specifies in
every one of the 21 stress periods, and in the first period it rises from 62,509
to 68,760 m³/d (2.21 to 2.43 million ft³/d). Had the callback never run, that
ratio would be 1.0000.

## Recap

In this notebook you:
- Used a `modflowapi` callback to change a model input - the recharge - while the simulation ran, without rewriting the input files.
- Confirmed the change from the model's own budget output, with no second run.

The callback is the easiest way to work with the standard stress packages; for data the package interface does not expose, the other API notebooks use lower-level memory access instead.